# Night Shift - approvals

What the agent drafted overnight and proved by replay, waiting for a person. Every row here is a `statement_repair` the policy gate quarantined: a replacement MERGE that graded replay_safe under late, duplicate, out-of-order, retried and corrected deliveries, where the original did not.

Approving does not change anyone's job. The next scheduled heartbeat tick re-evaluates the approval through the same gate (the verdict must still be fresh) and writes the draft as a runbook row with your name on it.

In [ ]:
# The approval under review: its id drives every cell below. Pick one from the Pending table in step 1.
import re as _re
APPROVAL_ID = ""
if APPROVAL_ID and not _re.fullmatch(r"[0-9a-f]{16}", APPROVAL_ID):
    raise ValueError("APPROVAL_ID must be the 16-hex-character id from the Pending table")


## 1. Pending

In [ ]:
spark.sql("""SELECT a.approval_id, get_json_object(a.action_json, '$.kind') AS kind, get_json_object(a.action_json, '$.target') AS target,
       d.verdict, d.diff_rows, d.model, a.requested_at, a.expires_at, a.note
FROM default.nightshift_ops.approvals a LEFT JOIN default.nightshift_ops.repair_drafts d ON d.approval_id = a.approval_id
WHERE a.status = 'pending' ORDER BY a.requested_at""").show(20, truncate=False)

## 2. The evidence: original and draft, same hazards, same table

In [ ]:
# The verdicts behind it: the original statement and the draft, same five hazards, each on its own shallow clone of the table.
_ops = "default.nightshift_ops"
_d = spark.sql(f"SELECT statement_sha256, verdict_id, table_name FROM {_ops}.repair_drafts WHERE approval_id = '{APPROVAL_ID}'").collect()
if not _d:
    raise ValueError(f"no draft row for approval {APPROVAL_ID}")
_sha, _vid, _table = _d[0]
print(_table)
_orig = spark.sql(f"SELECT 'original' AS which, verdict, diffs_json, elapsed_s FROM {_ops}.replay_verdicts WHERE statement_sha256 = '{_sha}' AND candidate NOT LIKE 'draft:%' ORDER BY created_at DESC LIMIT 1")
_draft = spark.sql(f"SELECT 'draft' AS which, verdict, diffs_json, elapsed_s FROM {_ops}.replay_verdicts WHERE verdict_id = '{_vid}'")
_orig.unionAll(_draft).show(truncate=False)


In [ ]:
# The draft itself, as text. It becomes a runbook procedure if approved; it is never applied to anyone's job.
spark.sql(f"SELECT draft_id, model, validation, verdict, draft_sql FROM default.nightshift_ops.repair_drafts WHERE approval_id = '{APPROVAL_ID}'").show(truncate=False)


## 3. Decide

In [ ]:
# Decide the approval named above. Set DECISION to "approve" or "reject", run the cell.
# The row is updated only if it is still pending; approved_by is the identity this kernel runs as.
DECISION = "approve"
NOTE = ""

_ops = "default.nightshift_ops"
_status = {"approve": "approved", "reject": "rejected"}[DECISION]
# Works on its own too: if the first cell was not run in this kernel, the id staged into this notebook is used.
APPROVAL_ID = globals().get("APPROVAL_ID") or ""
if not APPROVAL_ID:
    raise ValueError("set APPROVAL_ID in the first cell")
_who = spark.sql("SELECT current_user()").collect()[0][0]
# Spark SQL escaping for the literals below: backslash doubled, quote backslash-escaped. A doubled quote ('') is NOT an
# escape in Spark (it reads as two adjacent literals and the quote is lost), so the ledger's own rule is used here.
_esc = lambda s: s.replace("\\", "\\\\").replace("'", "\\'")  # noqa: E731
_who_sql = _esc(_who)
_note_sql = _esc(NOTE)
print(f"deciding {APPROVAL_ID} as {_who}")
_row = spark.sql(f"SELECT status FROM {_ops}.approvals WHERE approval_id = '{APPROVAL_ID}'").collect()
if not _row:
    raise ValueError(f"no approval {APPROVAL_ID}")
if _row[0][0] != "pending":
    raise ValueError(f"approval {APPROVAL_ID} is {_row[0][0]}, not pending; nothing changed")
_suffix = f"{_status} by {_who_sql}: workbench" + (f"; {_note_sql}" if NOTE else "")
spark.sql(f"""UPDATE {_ops}.approvals
SET status = '{_status}', approved_by = '{_who_sql}', approved_at = current_timestamp(),
    note = CASE WHEN note IS NULL THEN '{_suffix}' ELSE CONCAT(note, ' | ', '{_suffix}') END
WHERE approval_id = '{APPROVAL_ID}' AND status = 'pending'""")
spark.sql(f"SELECT approval_id, status, approved_by, approved_at, expires_at FROM {_ops}.approvals WHERE approval_id = '{APPROVAL_ID}'").show(truncate=False)
print("approved: the next scheduled heartbeat tick (within 30 minutes) re-checks it through the gate and writes the runbook row"
      if _status == "approved" else "rejected: recorded; the draft is never applied")


## 4. After the next tick

The runbook row carries who approved it and when, and the draft as its procedure.

In [ ]:
spark.sql("""SELECT approved_at, approved_by, label, title, provenance, procedure FROM default.nightshift_ops.runbooks ORDER BY approved_at DESC""").show(10, truncate=False)

In [ ]:
spark.sql("""SELECT approval_id, status, approved_by, approved_at, note FROM default.nightshift_ops.approvals ORDER BY requested_at DESC""").show(10, truncate=False)